In [1]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [4]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher,CurrentYearMonthDataFetcher
from src.ghcn_daily.data_processing import WeatherDataProcessor
import numpy as np

In [5]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')
latest_data = CurrentYearMonthDataFetcher(config_file="settings.json",data_type='dataframe')

In [6]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [7]:
s_state_list=stations[stations['STATE']=='NH']['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024)]['ID'].unique().tolist()

In [10]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data: 100%|████████████████████████████████████████████████| 16/16 [00:08<00:00,  1.85it/s]


In [11]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [12]:
processor=WeatherDataProcessor(data,weather_variables)

In [13]:
df=processor.process_data()

In [14]:
import pandas as pd
def list_stations_with_less_than_5_percent_missing(df):
    stations = df["ID"].unique()
    stations_with_less_than_5_percent_missing = []
    for station in stations:
        station_data = df[df["ID"] == station].copy()
        station_data.loc[:, 'DATE'] = pd.to_datetime(station_data['DATE'])
        station_data.sort_index(inplace=True)
        columns_to_check = ['TMIN', 'TMAX', 'SNOW', 'SNWD', 'PRCP']
        missing_percentage = station_data[columns_to_check].isnull().mean() * 100
        if (missing_percentage < 5).all():
            stations_with_less_than_5_percent_missing.append(station)
    return stations_with_less_than_5_percent_missing
list = list_stations_with_less_than_5_percent_missing(df)


In [16]:
df=df[df['ID'].isin(list)]

In [18]:
df = pd.merge(df, stations, on='ID', how='left')
df = df[['DATE','ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']]

In [21]:
from src.ghcn_daily.data_processing import  WeatherDataCleaner
cleaner = WeatherDataCleaner(df)

In [28]:
df=cleaner.clean_all(alpha=0.6)

In [33]:
df.to_csv('/workspaces/BlizzardX/Data/cleaned_data.csv', index=False)

In [40]:
from src.Model.feature_engineering import FeatureEngineering
fe=FeatureEngineering(df)

In [41]:
df=fe.apply_all_features()

AttributeError: 'FeatureEngineering' object has no attribute 'groupby'